<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 2 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">How a language model produces text · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to Chapter 2's practice challenges. Challenge 2 calls the model and replays its recorded answers, and challenges 1 and 3 run on the chapter's `sampling.py` alone.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording, and the module from the chapter's Claude Code build.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/how-a-language-model-produces-text--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("how-a-language-model-produces-text--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 10 recorded responses from claude-sonnet-5-5, recorded 2026-11-11. Set LIVE = True in Setup to call the API yourself.


In [2]:
# sampling.py is the module Claude Code generated in this chapter's build, from the spec and tests in
# builds/how-a-language-model-produces-text/. Colab fetches it from the book's repository.
BUILD = "how-a-language-model-produces-text"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/sampling.py", "sampling.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import sampling
importlib.reload(sampling)
from sampling import empirical_distribution, entropy_bits, softmax

In [3]:
MODEL = "claude-sonnet-5-5"
CANDIDATES = ["today", "yesterday", "late", "damaged", "safely"]
SCORES = [2.0, 1.6, 1.1, 0.4, 0.2]   # the chapter's illustrative scores


def answer_text(reply):
    """The text of a reply. Its content is a list of blocks, and the text blocks hold the answer."""
    return "".join(block.text for block in reply.content if block.type == "text")

## Challenge 1. Temperature at the extremes

Recompute the chapter's illustrative scores at temperature 0.1 and 10, and compare the entropy with the chapter's three temperatures.

In [4]:
# Challenge 1. The same scores at a very low and a very high temperature.
for t in [0.1, 10.0]:
    probs = softmax(SCORES, temperature=t)
    print(f"temperature {t}: " + "  ".join(f"{w} {p:.3f}" for w, p in zip(CANDIDATES, probs))
          + f"   entropy {entropy_bits(probs):.2f} bits")

temperature 0.1: today 0.982  yesterday 0.018  late 0.000  damaged 0.000  safely 0.000   entropy 0.13 bits
temperature 10.0: today 0.219  yesterday 0.211  late 0.200  damaged 0.187  safely 0.183   entropy 2.32 bits


At 0.1, *today* takes 0.982 of the probability and *yesterday* the remaining 0.018, an entropy of 0.13 bits, close to always choosing the top word. At 10 the five shares run from 0.219 down to 0.183, and the entropy rounds to the same 2.32 bits as five equal shares. A low temperature approaches a fixed choice, and a high one approaches a uniform draw.

## Challenge 2. Measure a new request

Send *Name a primary color. Reply with one word.* ten times and compute its distribution and entropy.

In [5]:
# Challenge 2. A new request, sent ten times.
PROMPT = "Name a primary color. Reply with one word."
answers = []
for _ in range(10):
    reply = client.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": PROMPT}])
    answers.append(answer_text(reply))
dist = empirical_distribution(answers)
print(", ".join(f"{answer} {share:.0%}" for answer, share in dist.items()), f"  entropy {entropy_bits(dist):.2f} bits")

blue 100%   entropy 0.00 bits


All ten answers were *blue*, an entropy of 0 bits. Like the chapter's random number, a request with several valid answers came back with one, which shows how the model ranks them and says nothing about whether the others are wrong.

## Challenge 3. Close the gap in the spec

Claude Code reported that the spec doesn't say whether whitespace left after removing trailing punctuation is stripped, so `"hello ."` normalizes to `"hello "`. Show the gap, then write the test that settles it.

In [6]:
# Challenge 3. The gap Claude Code reported in the spec: an answer with a space before its final period.
print(empirical_distribution(["hello", "hello ."]))

# A test that settles it. Under the generated code it fails, so the next build starts from this test.
def test_space_before_trailing_punctuation_is_removed():
    assert empirical_distribution(["hello", "hello ."]) == {"hello": 1.0}

try:
    test_space_before_trailing_punctuation_is_removed()
    print("pass")
except AssertionError:
    print("FAIL: 'hello .' is counted as a different answer from 'hello'")

{'hello': 0.5, 'hello ': 0.5}
FAIL: 'hello .' is counted as a different answer from 'hello'


The generated module counts `'hello'` and `'hello '` as two answers, each half of the total, and the new test fails. Adding it to the build's `given/tests/` and running the build again hands Claude Code the decision in a form it can check, and the spec should say the same in words.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>